<a href="https://colab.research.google.com/github/ashok-bisht/Context-Aware_Misinformation_Detection_ML_and_Gen_AI/blob/main/notebooks/1.0-eda-data-cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### 1. Load and Merge Data
First, I will load the `Fake.csv` and `True.csv` files into pandas DataFrames. I will add a `label` column to distinguish between fake (0) and real (1) news, and then concatenate them into a single DataFrame. The `text` column in both datasets will be renamed to `content` to standardize it.

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
import matplotlib.pyplot as plt
import seaborn as sns

# EDA

* Load and read df_true and df_fake data.
* View the description of the true and fake sets.
*  Label for the true and fake sets (1, 0).

## Clean the data:
* Concat the two datasets into df.
* Remove unused columns, keeping only the title and label.
* Remove missing rows with drop null.
* Remove extra spaces.
* Check if the number of real/fake records is equal.
* Mix the data to ensure training.
* Calculate the length of each title in a data point.
* Visualize.
* Select the appropriate MAX LENGTH based on the quantile of sentence length.



In [ ]:
import pandas as pd

# Load the datasets
df_fake = pd.read_csv('/content/Fake.csv')
df_true = pd.read_csv('/content/True.csv')

# Add a 'label' column (0 for fake, 1 for true)
df_fake['label'] = 0
df_true['label'] = 1

# Rename 'text' column to 'content' for consistency
df_fake.rename(columns={'text': 'content'}, inplace=True)
df_true.rename(columns={'text': 'content'}, inplace=True)

# Concatenate the dataframes
df_combined = pd.concat([df_fake, df_true], ignore_index=True)

# Display the first few rows and info of the combined dataframe
print("Combined DataFrame Head:")
display(df_combined.head())
print("\nCombined DataFrame Info:")
df_combined.info()
df_combined.describe()

Combined DataFrame Head:


,title,content,subject,date,label
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",0
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",0
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",0
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",0
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",0



Combined DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44898 entries, 0 to 44897
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    44898 non-null  object
 1   content  44898 non-null  object
 2   subject  44898 non-null  object
 3   date     44898 non-null  object
 4   label    44898 non-null  int64 
dtypes: int64(1), object(4)
memory usage: 1.7+ MB


,label
count,44898.000000
mean,0.477015
std,0.499477
min,0.000000
25%,0.000000
50%,0.000000
75%,1.000000
max,1.000000


In [ ]:
#print label==0, subject wise counts
print ("Fake News:")
print(df_combined[df_combined['label'] == 0]['subject'].value_counts())
print ("--"*50)
print ("True News:")
print(df_combined[df_combined['label'] == 1]['subject'].value_counts())
print (df_combined.shape)

Fake News:
subject
News               9050
politics           6841
left-news          4459
Government News    1570
US_News             783
Middle-east         778
Name: count, dtype: int64
----------------------------------------------------------------------------------------------------
True News:
subject
politicsNews    11272
worldnews       10145
Name: count, dtype: int64
(44898, 5)


In [ ]:
#Remove redundant columns

df_combined.drop(['date'], axis=1, inplace=True)
print (df_combined.shape)

(44898, 4)


In [ ]:
#Drop NA records
df_combined.dropna(inplace=True)
print ("After Drop NA",df_combined.shape)
#Trim spaces at end
df_combined['title'] = df_combined['title'].astype(str).str.strip()
df_combined['content'] = df_combined['content'].astype(str).str.strip()
df_combined['subject'] = df_combined['subject'].astype(str).str.strip()
print ("After Trim Space", df_combined.shape)
#remove the duplicate records
df_combined.drop_duplicates(inplace=True)
print ("After Duplicate removal", df_combined.shape)


After Drop NA (44898, 4)
After Trim Space (44898, 4)
After Duplicate removal (44682, 4)


In [ ]:
print ("Check the distribution of True and False")
print(df_combined["label"].value_counts())

Check the distribution of True and False
label
0    23475
1    21207
Name: count, dtype: int64


In [ ]:
# Randomly mix the data
df_combined = df_combined.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
df_combined.head()

,title,content,subject,label
0,"Florida Is About To Make Murder Legal – Yes, R...","Soon, if you live in Florida, it will become v...",News,0
1,Trump Campaign Gives PATHETIC Reason Why Georg...,"Months ago, former First Lady Laura Bush shock...",News,0
2,Mexico says upcoming U.S. execution of nationa...,MEXICO CITY (Reuters) - Senior Mexican diploma...,worldnews,1
3,HYSTERICAL…THE DEMOCRAT CONVENTION Schedule Is...,LOL! You ll want to share this with everyone D...,politics,0
4,You Really Don’t Want To Miss Ana Navarro PUMM...,When it comes to Donald Trump and his number o...,News,0


In [ ]:
# Install spaCy and download the English model
!pip install spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 114.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


### 2. Text Cleaning with spaCy

Now, I will clean the text data using spaCy. This involves:
-   **Loading the spaCy model**: `en_core_web_sm`.
-   **Tokenization**: Breaking down text into individual words.
-   **Stopword Removal**: Removing common words that don't add much meaning.
-   **Lowercasing**: Converting all text to lowercase.
-   **Removing Punctuation and Special Characters**.
-   **Lemmatization**: Reducing words to their base form.

After cleaning, I will save the processed DataFrame to a new CSV file named `cleaned_news_data.csv`.

In [ ]:
# Save the df_combined to this location
import os

# 1. Define the Google Drive folder and file name
drive_folder = '/content/drive/MyDrive/Colab Notebooks/Context-Aware Misinformation Detection/'
csv_path = os.path.join(drive_folder, 'combined_news.csv')

# 2. Ensure the directory exists
os.makedirs(drive_folder, exist_ok=True)

print("Saving DataFrame to Google Drive... Please wait.")

# 3. Save to CSV (index=False prevents pandas from adding an extra row numbers column)
df_combined.to_csv(csv_path, index=False)

print(f"🎉 Success! Dataset successfully saved to: {final_csv_path}")



Saving DataFrame to Google Drive... Please wait.
🎉 Success! Dataset successfully saved to: /content/drive/MyDrive/Colab Notebooks/Context-Aware Misinformation Detection/combined_news.csv


In [ ]:
#load the DF from the drive
import pandas as pd
#pd.set_option('display.max_colwidth', 50)
df_combined = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Context-Aware Misinformation Detection/combined_news.csv')
df_combined.head()

,title,content,subject,label
0,"Florida Is About To Make Murder Legal – Yes, R...","Soon, if you live in Florida, it will become v...",News,0
1,Trump Campaign Gives PATHETIC Reason Why Georg...,"Months ago, former First Lady Laura Bush shock...",News,0
2,Mexico says upcoming U.S. execution of nationa...,MEXICO CITY (Reuters) - Senior Mexican diploma...,worldnews,1
3,HYSTERICAL…THE DEMOCRAT CONVENTION Schedule Is...,LOL! You ll want to share this with everyone D...,politics,0
4,You Really Don’t Want To Miss Ana Navarro PUMM...,When it comes to Donald Trump and his number o...,News,0


In [ ]:
import os
from datetime import datetime
import pandas as pd
import spacy

# ==========================================
# ⚙️ CONFIGURATION BLOCK
# ==========================================
START_ROW = 0     # Change this to resume (e.g., 15000) if it fails midway
N = 1000          # Configurable 'n' rows per batch, CSV append, and log update
# ==========================================

# 1. Setup Folder and File Paths
drive_folder = '/content/drive/MyDrive/Colab Notebooks/Context-Aware Misinformation Detection/'
final_csv_path = os.path.join(drive_folder, 'cleaned_news_data.csv')

# 2. Load spaCy with optimized disabling to maximize performance
nlp = spacy.load('en_core_web_sm', disable=['tok2vec', 'parser', 'ner'])

# Reusable helper function to process a list of texts through nlp.pipe
def clean_text_list(text_list):
    cleaned = []
    for doc in nlp.pipe(text_list, batch_size=250, n_process=-1):
        tokens = [token.lemma_ for token in doc if token.is_alpha and not token.is_stop]
        cleaned.append(" ".join(tokens))
    return cleaned

total_rows = len(df_combined)
print(f"Starting processing from row {START_ROW} out of {total_rows} total rows (Batch size N = {N})...")

# 3. Process and Save in Configurable Batches
for start in range(START_ROW, total_rows, N):
    end = min(start + N, total_rows)

    # Slice the current chunk from the main DataFrame
    batch_df = df_combined.iloc[start:end].copy()

    # --- PROCESS COLUMN 1: TITLE ---
    title_texts = batch_df['title'].fillna("").astype(str).str.lower().tolist()
    batch_df['cleaned_title'] = clean_text_list(title_texts)

    # --- PROCESS COLUMN 2: CONTENT ---
    content_texts = batch_df['content'].fillna("").astype(str).str.lower().tolist()
    batch_df['cleaned_content'] = clean_text_list(content_texts)

    # --- PROCESS COLUMN 3: SUBJECT ---
    # Subject fields are usually single words/categories, cleaning keeps them uniform
    subject_texts = batch_df['subject'].fillna("").astype(str).str.lower().tolist()
    batch_df['cleaned_subject'] = clean_text_list(subject_texts)

    # 4. Save to CSV in Append Mode
    if start == 0 and not os.path.exists(final_csv_path):
        batch_df.to_csv(final_csv_path, index=False, mode='w')
    else:
        # Append mode ('a') bypasses writing the column headers again
        batch_df.to_csv(final_csv_path, index=False, mode='a', header=False)

    # 5. Generate Log File with Unique Timestamp Suffix
    current_time = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_filename = f"progress_log_{current_time}.txt"
    log_filepath = os.path.join(drive_folder, log_filename)

    log_content = (
        f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n"
        f"Status: Success\n"
        f"Last Completed Index: {end - 1}\n"
        f"Processed Rows: {start} to {end - 1}\n"
        f"Total Data Progress: {end}/{total_rows} rows complete.\n"
    )

    with open(log_filepath, "w") as log_file:
        log_file.write(log_content)

    print(f"✅ Appended rows {start} to {end-1} (Title, Content, and Subject cleaned). Log: {log_filename}")

print(f"\n🎉 All processing completed! Final file saved with multiple clean features at: {final_csv_path}")


Starting processing from row 0 out of 44682 total rows (Batch size N = 1000)...
✅ Appended rows 0 to 999 (Title, Content, and Subject cleaned). Log: progress_log_20260727_214918.txt
✅ Appended rows 1000 to 1999 (Title, Content, and Subject cleaned). Log: progress_log_20260727_214951.txt
✅ Appended rows 2000 to 2999 (Title, Content, and Subject cleaned). Log: progress_log_20260727_215021.txt
✅ Appended rows 3000 to 3999 (Title, Content, and Subject cleaned). Log: progress_log_20260727_215051.txt
✅ Appended rows 4000 to 4999 (Title, Content, and Subject cleaned). Log: progress_log_20260727_215122.txt
✅ Appended rows 5000 to 5999 (Title, Content, and Subject cleaned). Log: progress_log_20260727_215153.txt
✅ Appended rows 6000 to 6999 (Title, Content, and Subject cleaned). Log: progress_log_20260727_215223.txt
✅ Appended rows 7000 to 7999 (Title, Content, and Subject cleaned). Log: progress_log_20260727_215252.txt
✅ Appended rows 8000 to 8999 (Title, Content, and Subject cleaned). Log: pro

### 3. TF-IDF Vectorization

Finally, I will perform TF-IDF (Term Frequency-Inverse Document Frequency) vectorization on the `cleaned_content` column. TF-IDF is a numerical statistic that reflects how important a word is to a document in a collection or corpus.

I will use `TfidfVectorizer` from `sklearn.feature_extraction.text` to convert the text data into a matrix of TF-IDF features. The resulting TF-IDF matrix will be saved as a new CSV file named `tfidf_vectors.csv` for use by other team members.

In [4]:
import os
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
import scipy.sparse
import numpy as np

# 1. Load the cleaned data
cleaned_file_path = '/content/drive/MyDrive/Colab Notebooks/Context-Aware Misinformation Detection/cleaned_news_data.csv'
df_combined = pd.read_csv(cleaned_file_path)

# 2. Critical Step: Fill empty/missing cells with blank strings to prevent crashes
df_combined['cleaned_title'] = df_combined['cleaned_title'].fillna("")
df_combined['cleaned_content'] = df_combined['cleaned_content'].fillna("")
df_combined['cleaned_subject'] = df_combined['cleaned_subject'].fillna("")

# 3. Initialize separate vectorizers using a ColumnTransformer
# We apply different max_feature caps based on typical text length per field
preprocessor = ColumnTransformer(
    transformers=[
        ('title_tfidf', TfidfVectorizer(max_features=5000), 'cleaned_title'),
        ('content_tfidf', TfidfVectorizer(max_features=25000), 'cleaned_content'),
        ('subject_tfidf', TfidfVectorizer(max_features=500), 'cleaned_subject')
    ]
)

print("Vectorizing features in parallel...")
# 4. Transform your columns into a horizontal combined sparse matrix
tfidf_matrix = preprocessor.fit_transform(df_combined)

# 5. Extract unique, descriptive names for all newly generated feature columns
feature_names = preprocessor.get_feature_names_out()

# Define output file paths
drive_folder = '/content/drive/MyDrive/Colab Notebooks/Context-Aware Misinformation Detection/'
sparse_matrix_file_path = os.path.join(drive_folder, 'tfidf_sparse_matrix.npz')
feature_names_file_path = os.path.join(drive_folder, 'tfidf_feature_names.npy')
labels_file_path = os.path.join(drive_folder, 'tfidf_labels.csv')

# 6. Save the sparse matrix using scipy.sparse.save_npz
scipy.sparse.save_npz(sparse_matrix_file_path, tfidf_matrix)
print(f"\nSparse TF-IDF matrix successfully saved to: {sparse_matrix_file_path}")

# 7. Save feature names (as a numpy array)
np.save(feature_names_file_path, feature_names)
print(f"TF-IDF feature names successfully saved to: {feature_names_file_path}")

# 8. Save the classification labels separately
df_combined['label'].to_csv(labels_file_path, index=False)
print(f"Classification labels successfully saved to: {labels_file_path}")


Vectorizing features in parallel...

Sparse TF-IDF matrix successfully saved to: /content/drive/MyDrive/Colab Notebooks/Context-Aware Misinformation Detection/tfidf_sparse_matrix.npz
TF-IDF feature names successfully saved to: /content/drive/MyDrive/Colab Notebooks/Context-Aware Misinformation Detection/tfidf_feature_names.npy
Classification labels successfully saved to: /content/drive/MyDrive/Colab Notebooks/Context-Aware Misinformation Detection/tfidf_labels.csv
